In [1]:
# -*- coding: utf-8 -*-
"""
2D jointly trained BL-PINN with simultaneous phi/h/Q residual-based adaptive refinement.

Asymptotic system used in this script
--------------------------------------
Outer states:
    phi^pm * (phi^pm_x + phi^pm_y) = f(x,y).
Interface:
    h_t = 0.5 * (h_y - 1) * (phi^-(h,y) + phi^+(h,y)).
Inner corrections:
    Q^pm_xixi + [h_t + (phi^pm(h,y) + Q^pm)(1-h_y)]/sqrt(1+h_y^2) * Q^pm_xi = 0.
Matching at xi=0:
    phi^-(h,y)+Q^-(0) = phi^+(h,y)+Q^+(0) = 0.5*(phi^-(h,y)+phi^+(h,y)).
Far field (truncated):
    Q^-(-XI_MAX)=0 and Q^+(XI_MAX)=0, imposed as hard constraints.
Reconstruction coordinate:
    xi = (x-h(y,t))*sqrt(1+h_y(y,t)^2)/mu.
"""

from __future__ import annotations

import math
import os
import random
import time
from typing import Dict, List, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from scipy.spatial import cKDTree
from scipy.stats import qmc

Tensor = torch.Tensor
TensorInputs = Union[Tensor, Sequence[Tensor]]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

X_MIN, X_MAX = -2.0, 2.0
Y_MIN, Y_MAX = -4.0, 4.0
T_FINAL = 1.0
L_VALUE, R_VALUE = -4.0, 2.0
H0_VALUE = 0.0

DEPTH, WIDTH = 5, 10
LR = 1.0e-3
XI_MAX = 12.0
W_PHI_PER = 1.0
W_H_PER = 1.0

NUM_SAMPLES = 10000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200
BASE_PATH = "."
SAVE_LHS_PREDICTION = True
CHECK_EVERY = 100

PI = torch.tensor(math.pi, dtype=DTYPE, device=DEVICE)

# =============================================================================
# RAR Constants for Joint Training
# =============================================================================
METHOD_NAME = "BLPINN_RAR_PHI_H_Q"
N_PHI_P, N_H_P, N_M = 2000, 2000, 4000
INNER_LOSS_THRESHOLD = 1.0e-13
JOINT_INNER_MAX_STEPS = 20000

PHI_INITIAL_SIZE, PHI_ADD_K, PHI_MAX_RAR_BATCHES = 2600, 20, 20
PHI_CANDIDATE_SIZE, PHI_RESIDUAL_THRESHOLD = 35000, 1.0e-5

H_INITIAL_SIZE, H_ADD_K, H_MAX_RAR_BATCHES = 2600, 20, 20
H_CANDIDATE_SIZE, H_RESIDUAL_THRESHOLD = 35000, 1.0e-5

Q_INITIAL_SIZE, Q_ADD_K, Q_MAX_RAR_BATCHES = 3600, 20, 20
Q_CANDIDATE_SIZE, Q_RESIDUAL_THRESHOLD = 40000, 1.0e-5


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def mse(x: Tensor) -> Tensor:
    return torch.mean(x.square())

def mean_std(values: Sequence[float]) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1:
        return float(np.nanmean(arr)), 0.0
    return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1))

def all_grads(
    outputs: Tensor,
    inputs: TensorInputs,
    create_graph: bool = True,
    retain_graph: bool | None = None,
):
    if retain_graph is None:
        retain_graph = create_graph
    single = isinstance(inputs, torch.Tensor)
    input_tuple = (inputs,) if single else tuple(inputs)
    values = torch.autograd.grad(
        outputs=outputs,
        inputs=input_tuple,
        grad_outputs=torch.ones_like(outputs),
        create_graph=create_graph,
        retain_graph=retain_graph,
        only_inputs=True,
        allow_unused=False,
    )
    return values[0] if single else values

def source_f(x: Tensor, y: Tensor) -> Tensor:
    return torch.cos(PI * x / 4.0) * torch.cos(PI * y / 4.0)

class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, width: int, depth: int):
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                init.xavier_normal_(layer.weight)
                if layer.bias is not None:
                    init.zeros_(layer.bias)

    def forward(self, z: Tensor) -> Tensor:
        return self.net(z)

class ThreeModuleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.N_phi = MLP(2, 2, WIDTH, DEPTH)
        self.N_h = MLP(2, 1, WIDTH, DEPTH)
        self.N_Q = MLP(4, 2, WIDTH, DEPTH)

    def phi_m(self, x: Tensor, y: Tensor) -> Tensor:
        raw = self.N_phi(torch.cat([x, y], dim=1))[:, 0:1]
        return L_VALUE + (x - X_MIN) * raw

    def phi_p(self, x: Tensor, y: Tensor) -> Tensor:
        raw = self.N_phi(torch.cat([x, y], dim=1))[:, 1:2]
        return R_VALUE + (x - X_MAX) * raw

    def h(self, y: Tensor, t: Tensor) -> Tensor:
        return H0_VALUE + t * self.N_h(torch.cat([y, t], dim=1))

    def Q_m(self, xi: Tensor, h: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, y, t], dim=1))[:, 0:1]
        return ((xi + XI_MAX) / XI_MAX) * raw

    def Q_p(self, xi: Tensor, h: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, y, t], dim=1))[:, 1:2]
        return ((XI_MAX - xi) / XI_MAX) * raw

def set_trainable(model: ThreeModuleModel, phi: bool, h: bool, Q: bool) -> None:
    for p in model.N_phi.parameters(): p.requires_grad_(phi)
    for p in model.N_h.parameters(): p.requires_grad_(h)
    for p in model.N_Q.parameters(): p.requires_grad_(Q)

def sample_x(n: int) -> Tensor: return X_MIN + (X_MAX - X_MIN) * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_y(n: int) -> Tensor: return Y_MIN + (Y_MAX - Y_MIN) * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_t(n: int) -> Tensor: return T_FINAL * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_xi_m(n: int) -> Tensor: return -XI_MAX * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_xi_p(n: int) -> Tensor: return XI_MAX * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def constant_y(value: float, n: int) -> Tensor: return torch.full((n, 1), value, device=DEVICE, dtype=DTYPE)

# -----------------------------------------------------------------------------
# Derived losses
# -----------------------------------------------------------------------------
def outer_loss(model: ThreeModuleModel, x: Tensor, y: Tensor, x_per: Tensor):
    pm = model.phi_m(x, y)
    pp = model.phi_p(x, y)
    pm_x, pm_y = all_grads(pm, (x, y), create_graph=True, retain_graph=True)
    pp_x, pp_y = all_grads(pp, (x, y), create_graph=True, retain_graph=True)
    f = source_f(x, y)
    loss_res = mse(pm * (pm_x + pm_y) - f) + mse(pp * (pp_x + pp_y) - f)

    y_l = constant_y(Y_MIN, x_per.shape[0])
    y_r = constant_y(Y_MAX, x_per.shape[0])
    loss_per = mse(model.phi_m(x_per, y_l) - model.phi_m(x_per, y_r)) + mse(model.phi_p(x_per, y_l) - model.phi_p(x_per, y_r))
    return loss_res + W_PHI_PER * loss_per, loss_res, loss_per

def h_loss(model: ThreeModuleModel, y: Tensor, t: Tensor, t_per: Tensor):
    h_val = model.h(y, t)
    h_t, h_y = all_grads(h_val, (t, y), create_graph=True, retain_graph=True)
    pm_h = model.phi_m(h_val, y)
    pp_h = model.phi_p(h_val, y)
    residual = h_t - 0.5 * (h_y - 1.0) * (pm_h + pp_h)
    loss_res = mse(residual)

    y_l = constant_y(Y_MIN, t_per.shape[0])
    y_r = constant_y(Y_MAX, t_per.shape[0])
    loss_per = mse(model.h(y_l, t_per) - model.h(y_r, t_per))
    return loss_res + W_H_PER * loss_per, loss_res, loss_per

def frozen_h_data(model: ThreeModuleModel, y_base: Tensor, t_base: Tensor):
    y = y_base.detach().clone().requires_grad_(True)
    t = t_base.detach().clone().requires_grad_(True)
    h_val = model.h(y, t)
    h_y, h_t = all_grads(h_val, (y, t), create_graph=False, retain_graph=False)
    return h_val.detach(), h_y.detach(), h_t.detach()

def q_residual_frozen(
    model: ThreeModuleModel, side: str, xi_base: Tensor, y_base: Tensor, t_base: Tensor, training_graph: bool
) -> Tensor:
    h, h_y, h_t = frozen_h_data(model, y_base, t_base)
    y = y_base.detach()
    t = t_base.detach()
    phi = (model.phi_m(h, y) if side == "m" else model.phi_p(h, y)).detach()
    metric = torch.sqrt(1.0 + h_y.square())

    q_input = torch.cat([xi_base.detach(), h, y, t], dim=1).detach().requires_grad_(True)
    xi, h_q, y_q, t_q = q_input[:, 0:1], q_input[:, 1:2], q_input[:, 2:3], q_input[:, 3:4]
    Q = model.Q_m(xi, h_q, y_q, t_q) if side == "m" else model.Q_p(xi, h_q, y_q, t_q)

    q_grad = all_grads(Q, q_input, create_graph=True, retain_graph=True)
    Q_xi = q_grad[:, 0:1]
    Q_xixi = all_grads(Q_xi, q_input, create_graph=training_graph, retain_graph=training_graph)[:, 0:1]
    
    coeff = (h_t + (phi + Q) * (1.0 - h_y)) / metric
    residual = Q_xixi + coeff * Q_xi
    return residual if training_graph else residual.detach()

def q_residual_joint(model: ThreeModuleModel, side: str, xi: Tensor, y: Tensor, t: Tensor) -> Tensor:
    h = model.h(y, t)
    h_y, h_t = all_grads(h, (y, t), create_graph=True, retain_graph=True)
    phi = model.phi_m(h, y) if side == "m" else model.phi_p(h, y)
    metric = torch.sqrt(1.0 + h_y.square())
    Q = model.Q_m(xi, h, y, t) if side == "m" else model.Q_p(xi, h, y, t)
    
    Q_xi = all_grads(Q, xi, create_graph=True, retain_graph=True)
    Q_xixi = all_grads(Q_xi, xi, create_graph=True, retain_graph=True)
    coeff = (h_t + (phi + Q) * (1.0 - h_y)) / metric
    return Q_xixi + coeff * Q_xi

def q_match_joint(model: ThreeModuleModel, y: Tensor, t: Tensor) -> Tensor:
    h = model.h(y, t)
    pm = model.phi_m(h, y)
    pp = model.phi_p(h, y)
    middle = 0.5 * (pm + pp)
    xi0 = torch.zeros_like(t)
    qm = model.Q_m(xi0, h, y, t)
    qp = model.Q_p(xi0, h, y, t)
    return mse(pm + qm - middle) + mse(pp + qp - middle)

def q_loss_joint(
    model: ThreeModuleModel, xi_m: Tensor, y_m: Tensor, t_m: Tensor, xi_p: Tensor, y_p: Tensor, t_p: Tensor, y_match: Tensor, t_match: Tensor
):
    rm = q_residual_joint(model, "m", xi_m, y_m, t_m)
    rp = q_residual_joint(model, "p", xi_p, y_p, t_p)
    loss_res = mse(rm) + mse(rp)
    loss_match = q_match_joint(model, y_match, t_match)
    return loss_res + loss_match, loss_res, loss_match

# -----------------------------------------------------------------------------
# Reconstruction
# -----------------------------------------------------------------------------
def reconstruct(model: ThreeModuleModel, x: Tensor, y: Tensor, t: Tensor, mu: float) -> Tensor:
    with torch.enable_grad():
        y_in = y.detach().clone().requires_grad_(True)
        t_in = t.detach()
        h_val = model.h(y_in, t_in)
        h_y = all_grads(h_val, y_in, create_graph=False, retain_graph=False)

    with torch.no_grad():
        h = h_val.detach()
        metric = torch.sqrt(1.0 + h_y.detach().square())
        xi = (x - h) * metric / float(mu)
        xi_m = torch.clamp(xi, min=-XI_MAX, max=0.0)
        xi_p = torch.clamp(xi, min=0.0, max=XI_MAX)
        pm = model.phi_m(x, y)
        pp = model.phi_p(x, y)
        qm = model.Q_m(xi_m, h, y, t)
        qp = model.Q_p(xi_p, h, y, t)
        return torch.where(x <= h, pm + qm, pp + qp)

# -----------------------------------------------------------------------------
# Data Loading and Testing Set
# -----------------------------------------------------------------------------
def get_target_col(df: pd.DataFrame) -> str:
    if "u" in df.columns: return "u"
    if "u0" in df.columns: return "u0"
    return str(df.columns[-1])

def load_reference(mu: float):
    mu_id = round(-math.log10(mu))
    filename = f"2d_U0_all_t_u_x_y_t_mu{mu_id}_101_101_101_Mathematica_620.csv"
    path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(path): raise FileNotFoundError(f"Cannot find reference file: {filename}")
    df = pd.read_csv(path)
    df.columns = [str(c).lower().strip() for c in df.columns]
    df = df.sort_values(["t", "x", "y"]).reset_index(drop=True)
    return df, filename

def generate_lhs_indices(df: pd.DataFrame, mu: float) -> np.ndarray:
    mins, maxs = [df[c].min() for c in ["t","x","y"]], [df[c].max() for c in ["t","x","y"]]
    tree = cKDTree(df[["t", "x", "y"]].values)
    selected, used = [], set()
    for batch_id in range(100):
        sampler = qmc.LatinHypercube(d=3, seed=LHS_SEED + batch_id)
        points = qmc.scale(sampler.random(NUM_SAMPLES), mins, maxs)
        _, indices = tree.query(points)
        for idx in indices:
            idx = int(idx)
            if idx not in used:
                used.add(idx); selected.append(idx)
                if len(selected) == NUM_SAMPLES: break
        if len(selected) == NUM_SAMPLES: break
    if len(selected) < NUM_SAMPLES:
        remaining = np.setdiff1d(np.arange(len(df)), np.asarray(selected, dtype=int))
        fill = np.random.default_rng(LHS_SEED).choice(remaining, NUM_SAMPLES - len(selected), replace=False)
        selected.extend(int(v) for v in fill)
    result = np.asarray(selected, dtype=int)
    np.save(f"2d_LHS_sample_indices_mu{mu:.0e}.npy", result)
    return result

def build_lhs(mu: float) -> Dict[str, object]:
    df, filename = load_reference(mu)
    index_file = f"2d_LHS_sample_indices_mu{mu:.0e}.npy"
    indices = None
    if os.path.exists(index_file):
        loaded = np.load(index_file)
        if len(loaded) == NUM_SAMPLES and len(np.unique(loaded)) == NUM_SAMPLES:
            indices = loaded
    if indices is None:
        indices = generate_lhs_indices(df, mu)
    t_np = df.iloc[indices]["t"].to_numpy().reshape(-1, 1)
    x_np = df.iloc[indices]["x"].to_numpy().reshape(-1, 1)
    y_np = df.iloc[indices]["y"].to_numpy().reshape(-1, 1)
    u_np = df.iloc[indices][get_target_col(df)].to_numpy().reshape(-1)
    return {
        "t": torch.tensor(t_np, dtype=DTYPE, device=DEVICE),
        "x": torch.tensor(x_np, dtype=DTYPE, device=DEVICE),
        "y": torch.tensor(y_np, dtype=DTYPE, device=DEVICE),
        "true": u_np, "t_np": t_np, "x_np": x_np, "y_np": y_np, "n_test": len(indices),
    }

def errors(true: np.ndarray, pred: np.ndarray) -> Tuple[float, float]:
    diff = pred - true
    return float(np.linalg.norm(diff) / np.linalg.norm(true)), float(np.max(np.abs(diff)))

def timed_evaluation(model: ThreeModuleModel, data: Dict[str, object], mu: float):
    x, y, t = data["x"], data["y"], data["t"]
    for _ in range(EVAL_WARMUP): _ = reconstruct(model, x, y, t, mu)
    sync_cuda()
    start = time.perf_counter()
    for _ in range(EVAL_REPEAT): _ = reconstruct(model, x, y, t, mu)
    sync_cuda()
    t_eval = (time.perf_counter() - start) / EVAL_REPEAT
    pred = reconstruct(model, x, y, t, mu).detach().cpu().numpy().reshape(-1)
    e2, einf = errors(data["true"], pred)
    return t_eval, pred, e2, einf

def save_prediction(prefix: str, data: Dict[str, object], pred: np.ndarray) -> None:
    if SAVE_LHS_PREDICTION:
        pd.DataFrame({"t": data["t_np"].reshape(-1), "x": data["x_np"].reshape(-1), "y": data["y_np"].reshape(-1), "u": pred}).to_csv(prefix, index=False)

# =============================================================================
# Joint BL-PINN with simultaneous phi/h/Q RAR
# =============================================================================
def train_joint_rar(model, data):
    set_trainable(model, True, True, True)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    # Initialize point pools
    xphi, yphi = sample_x(PHI_INITIAL_SIZE), sample_y(PHI_INITIAL_SIZE)
    yh, th = sample_y(H_INITIAL_SIZE), sample_t(H_INITIAL_SIZE)
    xm, ym, tm = sample_xi_m(Q_INITIAL_SIZE), sample_y(Q_INITIAL_SIZE), sample_t(Q_INITIAL_SIZE)
    xp, yp, tp = sample_xi_p(Q_INITIAL_SIZE), sample_y(Q_INITIAL_SIZE), sample_t(Q_INITIAL_SIZE)
    
    phi_batches = h_batches = q_batches = total_steps = point_steps = candidate_evals = 0
    trace_phi = []; trace_h = []; trace_Q = []
    last_phi = last_h = last_Q = float("nan")
    last_phi_mean = last_h_mean = last_Q_mean = float("nan")
    
    sync_cuda()
    start = time.perf_counter()
    
    while True:
        n_inner = 0
        loss_value = float("inf")
        while n_inner < JOINT_INNER_MAX_STEPS and loss_value >= INNER_LOSS_THRESHOLD:
            optimizer.zero_grad(set_to_none=True)
            
            # Setup tensors with grad
            xphic = xphi.detach().clone().requires_grad_(True)
            yphic = yphi.detach().clone().requires_grad_(True)
            yhc, thc = yh.detach().clone().requires_grad_(True), th.detach().clone().requires_grad_(True)
            xmc, ymc, tmc = xm.detach().clone().requires_grad_(True), ym.detach().clone().requires_grad_(True), tm.detach().clone().requires_grad_(True)
            xpc, ypc, tpc = xp.detach().clone().requires_grad_(True), yp.detach().clone().requires_grad_(True), tp.detach().clone().requires_grad_(True)
            
            # Compute losses
            lphi, _, _ = outer_loss(model, xphic, yphic, data["xphi_p"])
            lh, _, _ = h_loss(model, yhc, thc, data["th_p"])
            lq, _, _ = q_loss_joint(model, xmc, ymc, tmc, xpc, ypc, tpc, data["ymatch"], data["tmatch"])
            
            loss = lphi + lh + lq
            loss.backward()
            optimizer.step()
            
            total_steps += 1
            n_inner += 1
            point_steps += int(xphi.shape[0]) + 2 * N_PHI_P + int(yh.shape[0]) + 2 * N_H_P + int(xm.shape[0] + xp.shape[0]) + N_M
            
            if n_inner % CHECK_EVERY == 0 or n_inner == JOINT_INNER_MAX_STEPS:
                last_phi, last_h, last_Q = float(lphi.detach().item()), float(lh.detach().item()), float(lq.detach().item())
                loss_value = last_phi + last_h + last_Q
                trace_phi.append(last_phi); trace_h.append(last_h); trace_Q.append(last_Q)

        # Checking caps
        phi_capped = phi_batches >= PHI_MAX_RAR_BATCHES
        h_capped = h_batches >= H_MAX_RAR_BATCHES
        q_capped = q_batches >= Q_MAX_RAR_BATCHES
        
        # Break AFTER training to ensure last added batches are optimized
        if phi_capped and h_capped and q_capped:
            break
            
        phi_added = h_added = q_added = False
        
        # Phi Candidate Search
        if not phi_capped:
            xc_phi = sample_x(PHI_CANDIDATE_SIZE).requires_grad_(True)
            yc_phi = sample_y(PHI_CANDIDATE_SIZE).requires_grad_(True)
            candidate_evals += PHI_CANDIDATE_SIZE
            
            pm_c, pp_c = model.phi_m(xc_phi, yc_phi), model.phi_p(xc_phi, yc_phi)
            pm_x, pm_y = all_grads(pm_c, (xc_phi, yc_phi), create_graph=False, retain_graph=False)
            pp_x, pp_y = all_grads(pp_c, (xc_phi, yc_phi), create_graph=False, retain_graph=False)
            
            with torch.no_grad():
                f_c = source_f(xc_phi, yc_phi)
                rm_phi = (pm_c.detach() * (pm_x.detach() + pm_y.detach()) - f_c).abs().reshape(-1)
                rp_phi = (pp_c.detach() * (pp_x.detach() + pp_y.detach()) - f_c).abs().reshape(-1)
                r_phi = 0.5 * (rm_phi + rp_phi)
                last_phi_mean = float(r_phi.mean().item())
                
            if last_phi_mean >= PHI_RESIDUAL_THRESHOLD:
                idx = torch.topk(r_phi, PHI_ADD_K).indices
                xphi = torch.cat([xphi, xc_phi[idx].detach()])
                yphi = torch.cat([yphi, yc_phi[idx].detach()])
                phi_batches += 1
                phi_added = True

        # H Candidate Search
        if not h_capped:
            yc = sample_y(H_CANDIDATE_SIZE).requires_grad_(True)
            tc = sample_t(H_CANDIDATE_SIZE).requires_grad_(True)
            candidate_evals += H_CANDIDATE_SIZE
            hv = model.h(yc, tc)
            ht, hy = all_grads(hv, (tc, yc), create_graph=False, retain_graph=False)
            
            with torch.no_grad():
                pm = model.phi_m(hv.detach(), yc.detach())
                pp = model.phi_p(hv.detach(), yc.detach())
                rh = (ht.detach() - 0.5 * (hy.detach() - 1.0) * (pm + pp)).abs().reshape(-1)
                last_h_mean = float(rh.mean().item())
                
            if last_h_mean >= H_RESIDUAL_THRESHOLD:
                idx = torch.topk(rh, H_ADD_K).indices
                yh = torch.cat([yh, yc[idx].detach()])
                th = torch.cat([th, tc[idx].detach()])
                h_batches += 1
                h_added = True
                
        # Q Candidate Search
        if not q_capped:
            xcm, ycm, tcm = sample_xi_m(Q_CANDIDATE_SIZE), sample_y(Q_CANDIDATE_SIZE), sample_t(Q_CANDIDATE_SIZE)
            xcp, ycp, tcp = sample_xi_p(Q_CANDIDATE_SIZE), sample_y(Q_CANDIDATE_SIZE), sample_t(Q_CANDIDATE_SIZE)
            candidate_evals += 2 * Q_CANDIDATE_SIZE
            
            rm = q_residual_frozen(model, "m", xcm, ycm, tcm, training_graph=False).abs().reshape(-1)
            rp = q_residual_frozen(model, "p", xcp, ycp, tcp, training_graph=False).abs().reshape(-1)
            last_Q_mean = float(0.5 * (rm.mean() + rp.mean()).item())
            
            if last_Q_mean >= Q_RESIDUAL_THRESHOLD:
                im = torch.topk(rm, Q_ADD_K).indices
                ip = torch.topk(rp, Q_ADD_K).indices
                xm = torch.cat([xm, xcm[im].detach()]); ym = torch.cat([ym, ycm[im].detach()]); tm = torch.cat([tm, tcm[im].detach()])
                xp = torch.cat([xp, xcp[ip].detach()]); yp = torch.cat([yp, ycp[ip].detach()]); tp = torch.cat([tp, tcp[ip].detach()])
                q_batches += 1
                q_added = True
                
        print(
            f"   [Joint RAR] b_phi={phi_batches}, b_h={h_batches}, b_Q={q_batches} | "
            f"m_phi={last_phi_mean:.3e}, m_h={last_h_mean:.3e}, m_Q={last_Q_mean:.3e}"
        )
        
        # Break if tolerances reached before max batches
        if not phi_added and not h_added and not q_added:
            break

    sync_cuda()
    return {
        "time": time.perf_counter() - start, "steps": total_steps, "point_steps": point_steps,
        "candidate_evals": candidate_evals, "phi_batches": phi_batches, "h_batches": h_batches, "Q_batches": q_batches,
        "phi_added": phi_batches * PHI_ADD_K, "h_added": h_batches * H_ADD_K, "Q_added": 2 * q_batches * Q_ADD_K,
        "final_phi": int(xphi.shape[0]), "final_h": int(yh.shape[0]), "final_Qm": int(xm.shape[0]), "final_Qp": int(xp.shape[0]),
        "last_phi": last_phi, "last_h": last_h, "last_Q": last_Q,
        "last_phi_mean": last_phi_mean, "last_h_mean": last_h_mean, "last_Q_mean": last_Q_mean,
        "trace_phi": np.asarray(trace_phi), "trace_h": np.asarray(trace_h), "trace_Q": np.asarray(trace_Q)
    }

def main():
    print("=" * 88)
    print("2D BL-PINN-RAR: Joint derived system, fully learning phi/h/Q with TRIPLE RAR")
    print(f"Device={DEVICE}, N_test={NUM_SAMPLES}")
    print("=" * 88)
    
    lhs = {mu: build_lhs(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}
    
    for seed in SEEDS:
        set_seed(seed)
        data = {
            "xphi_p": sample_x(N_PHI_P),
            "th_p": sample_t(N_H_P),
            "ymatch": sample_y(N_M),
            "tmatch": sample_t(N_M)
        }
        
        model = ThreeModuleModel().to(DEVICE)
        stats = train_joint_rar(model, data)
        
        np.save(f"2d_BLPINN_RAR_loss_phi_seed{seed}.npy", stats["trace_phi"])
        np.save(f"2d_BLPINN_RAR_loss_h_seed{seed}.npy", stats["trace_h"])
        np.save(f"2d_BLPINN_RAR_loss_Q_seed{seed}.npy", stats["trace_Q"])
        
        T_train = stats["time"]
        e_loss = stats["last_phi"] + stats["last_h"] + stats["last_Q"]
        
        print(
            f"seed={seed}: T_train={T_train:.2f}, loss={e_loss:.3e}, "
            f"phi_added={stats['phi_added']}, h_added={stats['h_added']}, Q_added={stats['Q_added']}"
        )
        
        model.eval()
        set_trainable(model, False, False, False)
        
        for mu in MU_LIST:
            T_eval, pred, e2, einf = timed_evaluation(model, lhs[mu], mu)
            
            metrics[mu].append({
                "Seed": seed, "N_test": lhs[mu]["n_test"],
                "loss_phi": stats["last_phi"], "loss_h": stats["last_h"], "loss_Q": stats["last_Q"],
                "e_loss": e_loss, "e2": e2, "einf": einf,
                "T_train": T_train, "T_eval": T_eval, "T_total": T_train + T_eval,
                "total_joint_iters": stats["steps"], "total_loss_point_steps": stats["point_steps"],
                "candidate_point_evals": stats["candidate_evals"],
                "phi_rar_batches": stats["phi_batches"], "h_rar_batches": stats["h_batches"], "Q_rar_batches": stats["Q_batches"],
                "phi_added": stats["phi_added"], "h_added": stats["h_added"], "Q_added_total": stats["Q_added"],
                "final_phi_points": stats["final_phi"], "final_h_points": stats["final_h"],
                "final_Q_minus_points": stats["final_Qm"], "final_Q_plus_points": stats["final_Qp"],
                "last_phi_mean_residual": stats["last_phi_mean"],
                "last_h_mean_residual": stats["last_h_mean"],
                "last_Q_mean_residual": stats["last_Q_mean"],
                "T_train_per_iter_ms": 1e3 * T_train / stats["steps"] if stats["steps"] else 0.0,
                "T_train_per_loss_point_us": 1e6 * T_train / stats["point_steps"] if stats["point_steps"] else 0.0
            })
            
            print(f"  mu={mu}: T_eval={T_eval:.6e}, e2={e2:.3e}, einf={einf:.3e}")
            save_prediction(f"2d_BLPINN_RAR_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv", lhs[mu], pred)
            
    for mu in MU_LIST:
        df = pd.DataFrame(metrics[mu])
        df.to_csv(f"2d_BLPINN_RAR_mu{mu:.0e}_Metrics_Summary.csv", index=False)
        print(f"\n### 2D BL-PINN-RAR, mu={mu} ###")
        for col in ["loss_phi", "loss_h", "loss_Q", "e2", "einf", "T_train", "T_eval", "T_total"]:
            m, s = mean_std(df[col])
            print(f"{col}: {m:.6e} +/- {s:.6e}")

if __name__ == "__main__":
    main()

2D BL-PINN-RAR: Joint derived system, fully learning phi/h/Q with TRIPLE RAR
Device=cuda, N_test=10000
   [Joint RAR] b_phi=1, b_h=1, b_Q=1 | m_phi=1.173e-02, m_h=5.785e-03, m_Q=5.721e-03
   [Joint RAR] b_phi=2, b_h=2, b_Q=2 | m_phi=9.664e-03, m_h=4.206e-03, m_Q=4.574e-03
   [Joint RAR] b_phi=3, b_h=3, b_Q=3 | m_phi=8.051e-03, m_h=4.011e-03, m_Q=4.243e-03
   [Joint RAR] b_phi=4, b_h=4, b_Q=4 | m_phi=6.524e-03, m_h=3.817e-03, m_Q=3.998e-03
   [Joint RAR] b_phi=5, b_h=5, b_Q=5 | m_phi=5.554e-03, m_h=3.894e-03, m_Q=4.028e-03
   [Joint RAR] b_phi=6, b_h=6, b_Q=6 | m_phi=5.208e-03, m_h=3.860e-03, m_Q=4.054e-03
   [Joint RAR] b_phi=7, b_h=7, b_Q=7 | m_phi=4.912e-03, m_h=3.632e-03, m_Q=3.849e-03
   [Joint RAR] b_phi=8, b_h=8, b_Q=8 | m_phi=4.709e-03, m_h=3.489e-03, m_Q=3.784e-03
   [Joint RAR] b_phi=9, b_h=9, b_Q=9 | m_phi=4.506e-03, m_h=3.188e-03, m_Q=3.703e-03
   [Joint RAR] b_phi=10, b_h=10, b_Q=10 | m_phi=4.599e-03, m_h=3.050e-03, m_Q=3.748e-03
   [Joint RAR] b_phi=11, b_h=11, b_Q=11 | m_